# Table 1 — Poisson Estimates of Yearly Privacy Patent Counts on Firm-Level Topic Relevance

| Item | Description |
|---|---|
| Goal | Do firms whose 10-K business description is closer to the consumer data privacy policy debate file more privacy patents per year than other firms of the same NAICS two-digit code in the same year, given firm age, size, and R&D? One specification: `ppmlhdfe n_privacy_patents relevance controls, absorb(naics2#fyear) vce(cluster firm_id)`. |
| Comparison | Across firms: a firm with a higher relevance score against firms with a lower score in the same NAICS two-digit code and fiscal year, on the number of privacy patents filed that year. |
| Unit and sample | Firm-year: the T4.1.3 analysis sample (689 firms of the all-years T3 sample with at least one patent filed in 1999 to 2017) in the fiscal years 1999 to 2021 in which they filed a 10-K, with the three lagged controls observed (T4.1.5). The Poisson estimator drops NAICS-year cells whose count is zero throughout and singleton cells; the surviving observations and firms are reported in the table. |
| Treated group / exposure | Continuous exposure: the firm's topic-relevance score, the cosine similarity between a firm-level summary of its 10-K Item 1 sections and the policy summary of legislator statements on consumer data privacy legislation (T3.8.3; the measure of T3.9.1 to T3.9.4 and T4.1.3), one score per firm, standardized to mean zero and standard deviation one over the regression firm-years. There is no binary treated group. |
| Control / reference group | Lower-score firms in the same NAICS two-digit code and year. Regression controls, all at t − 1: log firm age (years since the first CRSP listing), log total assets, log of one plus R&D expense with an indicator for missing R&D. |
| Timing and baseline | Fiscal years 1999 to 2021; the count of year t is the count filed in calendar year t; controls are lagged one fiscal year; no policy event or pre-post split. |
| Outcome | Count of KPSS patents filed in the year that carry at least one CPC code of the project's privacy code list (T4.1.2). Zeros are kept; the count is not transformed. |
| Estimate and interpretation | The coefficient on standardized relevance is the difference in the log of the expected privacy patent count between firms one standard deviation apart in relevance, within NAICS-year; exp(coefficient) minus one is the percent difference. |
| Adjustment and inference | `ppmlhdfe`, Poisson pseudo-maximum likelihood, with NAICS two-digit-by-year fixed effects absorbed; standard errors clustered by firm, the level at which the score is measured and the firm's years are correlated. Column 1 has no controls, column 2 the three lagged controls. |
| Assumption and limits | Descriptive cross-firm association. A causal reading needs that, within NAICS-year and given the controls, firms with different scores would otherwise file privacy patents at the same rate; business lines not captured by the two-digit code are the obvious omitted factor. The score is constant within firm, so a firm fixed effect cannot be added: `absorb(firm_id)` would absorb the score exactly and `ppmlhdfe` would drop it. A firm-fixed-effect design needs either a time-varying relevance measure or the score interacted with a period indicator. |
| Comparison status | The user asked for `ppmlhdfe privacy_patent_count topic_relevance controls, a(NAICS2#Year Gvkey)` with the firm-level score. The firm fixed effect is left out for the reason above; everything else follows the request. Open: R&D is entered as log of one plus the expense with a missing flag rather than dropping firm-years without reported R&D. |
| Input | `Processed/privacy_regression_panel.dta` from T4.1.5: one row per firm and fiscal year with the count, the firm-level score, the NAICS two-digit code, and the lagged controls. |
| Output | Table 1 as Markdown in this notebook and as `table_1_ppml_privacy_patents_topic_relevance.{tex,csv,md}` in `output/X1.1.6`. |


## Step 1. Set the Stata session options

A fresh session: the Stata version, no leftover data or estimates, no paging, and a wide line so that regression output is not wrapped.


In [0]:
version 18.5
clear all
set more off
set linesize 255


## Step 2. Define inputs and output locations

The two roots and the S5 folders as globals. The panel is read from the stage folder's `Processed` folder on scratch; the table goes to this notebook's output folder.


In [1]:
global PROJECT_FOLDER "/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis"
global SCRATCH_FOLDER "/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis"
global NOTEBOOK_SECTION "X1 - Example of Research Enabled by LLM Pipeline"
global NOTEBOOK_SUBFOLDER "${NOTEBOOK_SECTION}/X1.1 - Privacy Patents and Topic Relevance"
global NOTEBOOK_NAME "T4.1.6"
global PROCESSED_FOLDER "${SCRATCH_FOLDER}/${NOTEBOOK_SUBFOLDER}/Processed"
global TABLE_FOLDER "${PROJECT_FOLDER}/${NOTEBOOK_SECTION}/output/${NOTEBOOK_NAME}"
global PANEL_FILE "${PROCESSED_FOLDER}/privacy_regression_panel.dta"
capture mkdir "${TABLE_FOLDER}"
display "Panel: ${PANEL_FILE}"
display "Tables: ${TABLE_FOLDER}"


Panel: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/T4.1 - Privacy Patents and Topic Relevance/Processed/privacy_regression_panel.dta
Tables: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/output/T4.1.6


## Step 3. Load the panel and check the observation unit

One row per firm and fiscal year; the first rows are shown with the variables the regression uses.


In [2]:
use "${PANEL_FILE}", clear
isid gvkey fyear
count
list gvkey fyear naics2 n_privacy_patents relevance_firm ln_age_l1 ln_at_l1 ln_xrd_l1 rd_missing_l1 in 1/5, abbreviate(18) noobs


  11,464

  +------------------------------------------------------------------------------------------------------------------+
  |  gvkey   fyear   naics2   n_privacy_patents   relevance_firm   ln_age_l1    ln_at_l1   ln_xrd_l1   rd_missing_l1 |
  |------------------------------------------------------------------------------------------------------------------|
  | 001013    1999       33                   1          .522551   2.9957323    7.170571   4.9338406               0 |
  | 001013    2000       33                   1          .522551   3.0445224   7.4220921   5.6435794               0 |
  | 001013    2001       33                   0          .522551   3.0910425   8.2866473   5.8260001               0 |
  | 001013    2002       33                   0          .522551   3.1354942    7.823926     5.63336               0 |
  | 001013    2003       33                   0          .522551   3.1780538    7.042461   5.2694034               0 |
  +-----------------------------------

## Step 4. Restrict the sample and define the regression variables

Firm-years with all three lagged controls observed. The firm-level score is standardized over these firm-years so that one unit is one standard deviation. Numeric ids are built for the firm (the cluster) and for the NAICS two-digit-by-year cell (the fixed effect). The surviving rows, firms, and cells are reported, with a summary of the regression variables.


In [3]:
drop if missing(relevance_firm, ln_age_l1, ln_at_l1, ln_xrd_l1)
count
egen firm_id = group(gvkey)
egen naics2_year = group(naics2 fyear)
egen relevance_z = std(relevance_firm)
label variable relevance_z "Firm-level topic relevance, standardized over the regression firm-years"
label variable firm_id "Firm id"
label variable naics2_year "NAICS two-digit-by-year cell id"
egen firm_tag = tag(gvkey)
quietly count if firm_tag
display "Firms: " r(N)
quietly summarize naics2_year
display "NAICS two-digit-by-year cells: " r(max)
quietly summarize relevance_firm
display "Standard deviation of the firm-level score over the regression firm-years: " %6.4f r(sd)
summarize n_privacy_patents relevance_firm relevance_z ln_age_l1 ln_at_l1 ln_xrd_l1 rd_missing_l1


(58 observations deleted)
  11,406


Firms: 685
NAICS two-digit-by-year cells: 521
Standard deviation of the firm-level score over the regression firm-years: 0.0353

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
n_privacy_~s |     11,406    2.827547    17.85006          0        586
relevance_~m |     11,406    .5155057    .0353175    .405546    .620883
 relevance_z |

     11,406    6.70e-10           1  -3.113457    2.98371
   ln_age_l1 |     11,406    3.121181    .9266316          0   4.276666
    ln_at_l1 |     11,406    9.000192    1.712154   2.067369   15.19824
-------------+---------------------------------------------------------
   ln_xrd_l1 |     11,406    3.038199    2.843513          0   10.66291
rd_missing~1 |     11,406    .3570051    .4791373          0          1


## Step 5. Column (1): privacy patents on firm-level topic relevance, no controls

What this cell estimates: the cross-firm association between a one-standard-deviation higher relevance score and the number of privacy patents filed in the year, within NAICS two-digit-by-year cells and with no controls. The identifying assumption for a causal reading is that, within a cell, firms with different scores would otherwise file privacy patents at the same rate. `naics2_year` absorbs every shock common to a two-digit code in a year, including the code's overall patenting level. The firm fixed effect of the requested specification is not included because the score is constant within firm and would be absorbed by it. Standard errors are clustered by firm because the score is a firm-level variable repeated over the firm's years. Cells whose count is zero throughout drop out of the Poisson likelihood; the surviving observations and firms are retained for the table.


In [4]:
eststo clear
ppmlhdfe n_privacy_patents relevance_z, absorb(naics2_year) vce(cluster firm_id)
summarize `e(depvar)' if e(sample)
estadd scalar mean_dv = r(mean)
estadd scalar clusters = e(N_clust)
estadd local controls "No"
estadd local firm_fe "No"
estadd local naics2_year_fe "Yes"
eststo col_1


(dropped 2096 observations that are either singletons or separated by a fixed effect)


Iteration 1:   deviance = 1.4978e+05  eps = .         iters = 1    tol = 1.0e-04  min(eta) =  -5.25  P   
Iteration 2:   deviance = 1.0527e+05  eps = 4.23e-01  iters = 1    tol = 1.0e-04  min(eta) =  -6.18      


Iteration 3:   deviance = 9.8010e+04  eps = 7.41e-02  iters = 1    tol = 1.0e-04  min(eta) =  -7.06      
Iteration 4:   deviance = 9.7481e+04  eps = 5.43e-03  iters = 1    tol = 1.0e-04  min(eta) =  -8.03      


Iteration 5:   deviance = 9.7459e+04  eps = 2.24e-04  iters = 1    tol = 1.0e-04  min(eta) =  -8.77      
Iteration 6:   deviance = 9.7458e+04  eps = 1.63e-05  iters = 1    tol = 1.0e-04  min(eta) =  -9.22      
Iteration 7:   deviance = 9.7457e+04  eps = 5.80e-07  iters = 1    tol = 1.0e-05  min(eta) =  -9.36      


Iteration 8:   deviance = 9.7457e+04  eps = 2.25e-09  iters = 1    tol = 1.0e-06  min(eta) =  -9.37   S O
------------------------------------------------------------------------------------------------------------
(legend: p: exact partial-out   s: exact solver   h: step-halving   o: epsilon below tolerance)
Converged in 8 iterations and 8 HDFE sub-iterations (tol = 1.0e-08)



HDFE PPML regression                              No. of obs      =      9,310
Absorbing 1 HDFE group                            Residual df     =        651
Statistics robust to heteroskedasticity           Wald chi2(1)    =      17.95
Deviance             =  97457.45133               Prob > chi2     =     0.0000
Log pseudolikelihood = -52162.50022               Pseudo R2       =     0.4349

Number of clusters (firm_id)=        652


... (4 outputs left out)

------------------------------------------------------------------------------

Absorbed degrees of freedom:


-----------------------------------------------------+
 Absorbed FE | Categories  - Redundant  = Num. Coefs |


-------------+---------------------------------------|
 naics2_year |       249           0         249     |
-----------------------------------------------------+



    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
n_privacy_~s |      9,310    3.454672    19.70093          0        586

added scalar:
            e(mean_dv) =  3.4546724

added scalar:
           e(clusters) =  652

added macro:
           e(controls) : "No"

added macro:
            e(firm_fe) : "No"

added macro:
     e(naics2_year_fe) : "Yes"


## Step 6. Column (2): privacy patents on firm-level topic relevance with the lagged controls

What this cell estimates: the same association conditional on log firm age, log total assets, and log of one plus R&D expense at t − 1 (with the missing-R&D flag), so that older, larger, or more R&D-intensive firms are not read as more relevant firms. The rest is as in column (1). What would falsify the reading that relevance tracks privacy patenting: a coefficient near zero with a tight interval once the controls are in.


In [5]:
ppmlhdfe n_privacy_patents relevance_z ln_age_l1 ln_at_l1 ln_xrd_l1 rd_missing_l1, absorb(naics2_year) vce(cluster firm_id)
summarize `e(depvar)' if e(sample)
estadd scalar mean_dv = r(mean)
estadd scalar clusters = e(N_clust)
estadd local controls "Yes"
estadd local firm_fe "No"
estadd local naics2_year_fe "Yes"
eststo col_2


(dropped 2096 observations that are either singletons or separated by a fixed effect)


Iteration 1:   deviance = 6.6688e+04  eps = .         iters = 1    tol = 1.0e-04  min(eta) =  -6.51  P   


Iteration 2:   deviance = 5.0328e+04  eps = 3.25e-01  iters = 1    tol = 1.0e-04  min(eta) =  -7.83      


Iteration 3:   deviance = 4.7858e+04  eps = 5.16e-02  iters = 1    tol = 1.0e-04  min(eta) =  -9.25      


Iteration 4:   deviance = 4.7515e+04  eps = 7.21e-03  iters = 1    tol = 1.0e-04  min(eta) = -10.31      


Iteration 5:   deviance = 4.7457e+04  eps = 1.21e-03  iters = 1    tol = 1.0e-04  min(eta) = -11.17      


Iteration 6:   deviance = 4.7450e+04  eps = 1.50e-04  iters = 1    tol = 1.0e-04  min(eta) = -11.82      


Iteration 7:   deviance = 4.7450e+04  eps = 1.05e-05  iters = 1    tol = 1.0e-04  min(eta) = -12.16      


Iteration 8:   deviance = 4.7450e+04  eps = 2.17e-07  iters = 1    tol = 1.0e-05  min(eta) = -12.23   S  


Iteration 9:   deviance = 4.7450e+04  eps = 2.08e-10  iters = 1    tol = 1.0e-06  min(eta) = -12.24   S O
------------------------------------------------------------------------------------------------------------
(legend: p: exact partial-out   s: exact solver   h: step-halving   o: epsilon below tolerance)
Converged in 9 iterations and 9 HDFE sub-iterations (tol = 1.0e-08)



HDFE PPML regression                              No. of obs      =      9,310
Absorbing 1 HDFE group                            Residual df     =        651
Statistics robust to heteroskedasticity           Wald chi2(5)    =     324.72
Deviance             =  47449.86441               Prob > chi2     =     0.0000
Log pseudolikelihood = -27158.70676               Pseudo R2       =     0.7058

Number of clusters (firm_id)=        652
                               (Std. err. adjusted for 652 clusters in firm_id)
-------------------------------------------------------------------------------
              |               Robust
n_privacy_p~s | Coefficient  std. err.      z    P>|z|     [95% conf. interval]
--------------+----------------------------------------------------------------
  relevance_z |   .4753304   .1857105     2.56   0.010     .1113445    .8393163
    ln_age_l1 |   .2542147   .3181902     0.80   0.424    -.3694267    .8778561
     ln_at_l1 |   .3761897   .0929969     4.0

## Step 7. Assemble Table 1, display it, and export it

The two stored columns become one S4 table: the coefficient of standardized relevance and of each control with its clustered standard error below and stars at the 10, 5, and 1 percent levels, the specification rows, and the summary rows (observations, mean of the dependent variable on the estimation sample, the pseudo R² that `ppmlhdfe` reports, and the number of firm clusters). Both columns share one outcome, so it is named in the Notes rather than in a dependent-variable row. The table is written as Markdown, LaTeX, and CSV under one file stem and the Markdown is printed here with its title, Notes, and Source.


In [6]:
local models col_1 col_2
local n_models : word count `models'
local vars relevance_z ln_age_l1 ln_at_l1 ln_xrd_l1 rd_missing_l1
local lab_relevance_z "Topic relevance, firm level (standard deviations)"
local lab_ln_age_l1 "Log firm age, t − 1"
local lab_ln_at_l1 "Log total assets, t − 1"
local lab_ln_xrd_l1 "Log of one plus R&D expense, t − 1"
local lab_rd_missing_l1 "R&D expense missing, t − 1"
local table_title "Table 1—Poisson Estimates of Yearly Privacy Patent Counts on Firm-Level Topic Relevance"
local table_notes "The unit of observation is a firm-year: the T4.1.3 analysis sample (firms of the all-years T3 sample with at least one patent filed in 1999 to 2017), fiscal years 1999 to 2021 with a 10-K and the lagged controls observed. The dependent variable in both columns is the number of KPSS patents filed in the year that carry at least one CPC code of the project's privacy code list (T4.1.2). Each column is a Poisson pseudo-maximum-likelihood regression (ppmlhdfe) with NAICS two-digit-by-year fixed effects absorbed; cells whose count is zero throughout drop out. Topic relevance is the firm's score, the cosine similarity between a firm-level summary of its 10-K Item 1 sections and the policy summary of legislator statements on consumer data privacy legislation, one score per firm, standardized to mean zero and standard deviation one over the regression firm-years; a coefficient is the difference in the log expected count between firms one standard deviation apart. Firm fixed effects are not included because the score is constant within firm and would be absorbed by them. Controls are measured at fiscal year t − 1: firm age is years since the first CRSP listing; R&D means research and development, entered as the log of one plus the expense with missing R&D set to zero and flagged. Standard errors clustered by firm are in parentheses; * p < 0.10, ** p < 0.05, *** p < 0.01. Pseudo R² is the one ppmlhdfe reports."
local table_source "KPSS extended patent data (Kogan, Papanikolaou, Seru, and Stoffman, 2017, updated to 2025) with CPC codes; the privacy CPC code list in privacy_cpc/privacy_cpc_codes.xlsx; SEC EDGAR 10-K filings; congressional hearing statements from GovInfo; Compustat; CRSP-Compustat link; authors' calculations."
local table_name "table_1_ppml_privacy_patents_topic_relevance"

quietly {
preserve
clear
set obs 0
gen str80 label = ""
forvalues k = 1/`n_models' {
    gen str40 c`k' = ""
}
foreach v of local vars {
    local r = _N + 1
    set obs `=_N + 2'
    replace label = "`lab_`v''" in `r'
    local k = 0
    foreach m of local models {
        local ++k
        estimates restore `m'
        local col = colnumb(e(b), "`v'")
        if `col' < . {
            local b = _b[`v']
            local se = _se[`v']
            local p = 2 * normal(-abs(`b' / `se'))
            local stars = cond(`p' < 0.01, "***", cond(`p' < 0.05, "**", cond(`p' < 0.10, "*", "")))
            local bs = subinstr(string(`b', "%9.3f"), "-", "−", 1)
            local ses = subinstr(string(`se', "%9.3f"), "-", "−", 1)
            replace c`k' = "`bs'`stars'" in `r'
            replace c`k' = "(`ses')" in `=`r' + 1'
        }
    }
}
foreach row in "Lagged controls|controls" "Firm fixed effects|firm_fe" "NAICS two-digit-by-year fixed effects|naics2_year_fe" {
    local text = substr("`row'", 1, strpos("`row'", "|") - 1)
    local macro = substr("`row'", strpos("`row'", "|") + 1, .)
    local r = _N + 1
    set obs `r'
    replace label = "`text'" in `r'
    local k = 0
    foreach m of local models {
        local ++k
        estimates restore `m'
        replace c`k' = "`e(`macro')'" in `r'
    }
}
foreach row in "Observations|N|%12.0fc" "Mean of dependent variable|mean_dv|%9.3f" "Pseudo R²|r2_p|%9.2f" "Number of clusters (firms)|clusters|%12.0fc" {
    tokenize "`row'", parse("|")
    local text "`1'"
    local scalar "`3'"
    local format "`5'"
    local r = _N + 1
    set obs `r'
    replace label = "`text'" in `r'
    local k = 0
    foreach m of local models {
        local ++k
        estimates restore `m'
        replace c`k' = trim(string(e(`scalar'), "`format'")) in `r'
    }
}

* Markdown
file open md using "${TABLE_FOLDER}/`table_name'.md", write replace text
file write md "**`table_title'**" _n _n
file write md "| | (1) | (2) |" _n
file write md "|:---|:---:|:---:|" _n
forvalues i = 1/`=_N' {
    file write md "| `=label[`i']' | `=c1[`i']' | `=c2[`i']' |" _n
}
file write md _n "*Notes*: `table_notes'" _n _n "*Source*: `table_source'" _n
file close md

* LaTeX
local title_tex = subinstr("`table_title'", "—", "---", 1)
local notes_tex = subinstr("`table_notes'", "&", "\&", .)
file open tex using "${TABLE_FOLDER}/`table_name'.tex", write replace text
file write tex "\begin{table}[htbp]" _n "\centering" _n "\textbf{`title_tex'}\par" _n
file write tex "\begin{tabular}{lcc}" _n "\toprule" _n " & (1) & (2) \\" _n "\midrule" _n
forvalues i = 1/`=_N' {
    local l = label[`i']
    local l = subinstr("`l'", "&", "\&", .)
    local l = subinstr("`l'", "R²", "R\textsuperscript{2}", 1)
    if "`l'" == "Lagged controls" | "`l'" == "Observations" {
        file write tex "\midrule" _n
    }
    file write tex "`l' & `=c1[`i']' & `=c2[`i']' \\" _n
}
file write tex "\bottomrule" _n "\end{tabular}" _n
file write tex "\par\textit{Notes}: `notes_tex'" _n
file write tex "\par\textit{Source}: `table_source'" _n "\end{table}" _n
file close tex

* CSV
insobs 1, before(1)
replace label = "" in 1
forvalues k = 1/`n_models' {
    replace c`k' = "(`k')" in 1
}
export delimited label c1 c2 using "${TABLE_FOLDER}/`table_name'.csv", novarnames replace
restore
}

type "${TABLE_FOLDER}/`table_name'.md"
display _n "Saved: ${TABLE_FOLDER}/`table_name'.md, .tex, and .csv"


(Note: Below code run with echo to enable preserve/restore functionality.)
















**Table 1—Poisson Estimates of Yearly Privacy Patent Counts on Firm-Level Topic Relevance**

| | (1) | (2) |
|:---|:---:|:---:|
| Topic relevance, firm level (standard deviations) | 1.010*** | 0.475** |
|  | (0.238) | (0.186) |
| Log firm age, t − 1 |  | 0.254 |
|  |  | (0.318) |
| Log total assets, t − 1 |  | 0.376*** |
|  |  | (0.093) |
| Log of one plus R&D expense, t − 1 |  | 0.512*** |
|  |  | (0.131) |
| R&D expense missing, t − 1 |  | 1.831** |
|  |  | (0.861) |
| Lagged controls | No | Yes |
| Firm fixed effects | No | No |
| NAICS two-digit-by-year fixed effects | Yes | Yes |
| Observations | 9,310 | 9,310 |
| Mean of dependent variable | 3.455 | 3.455 |
| Pseudo R² | 0.43 | 0.71 |
| Number of clusters (firms) | 652 | 652 |

*Notes*: The unit of observation is a firm-year: the T4.1.3 analysis sample (firms of the all-years T3 sample with at least one patent filed in 1999 to 2017), fiscal years 1999 to 2021 with a 10-K and the lagged controls observed. The dependent variable i